<a href="https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.
30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

#**Task type: Ranking / Scoring**

My lane (CTR / Engagement Opportunity Scoring) is not a yes/no problem, so it is
not classification. Whether a page is an "opportunity" is not binary — the gap
between a page's actual CTR and the CTR expected for its position tier is a
continuous, graded signal: some pages sit barely below expectation, others sit
far below it. Picking one fixed cutoff (e.g. `ctr < 0.5`) would collapse that
gradient into an arbitrary yes/no split, and it wouldn't match the decision I'm
actually supporting: "which page should the editor review first?" — that needs
an ordered list, not a flag. So the natural mapping is ranking/scoring: score
every page by its opportunity gap, then rank so limited reviewer time goes to
the top of the list first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

#**Proxy: impressions-weighted CTR opportunity gap — engineered, not observed as a ready-made column**

The raw signals (`ctr`, `position_tier`, `impressions_90d`) are observed: real, measured
columns in the dataset. But the score I actually rank by does not exist as-is — I build it
myself in two steps.

##**Step 1 — the raw gap:**
`expected_ctr(tier) = mean(ctr) across all pages in that position_tier
opportunity_gap = expected_ctr(tier) − ctr.`

A positive gap means the page underperforms what's normal for its own tier (bigger gap =
bigger opportunity). This stays a continuous score, not a yes/no cutoff — consistent with
the Ranking/Scoring framing from section 1.

**Step 2 — why the raw gap alone isn't enough:**
Checking `opportunity_gap`'s unique values showed heavy repetition — 4,038 pages shared the
exact same gap value. All of them had `ctr = 0`, which collapses `opportunity_gap` to a
constant (`expected_ctr(tier)`) regardless of how many times the page was actually seen. A
page with 5 impressions and a page with 5,000 impressions — both at `ctr = 0` — would get
identical scores, even though the second is a far more reliable, higher-volume opportunity.

##**Fix — weight the gap by traffic volume:**
`weighted_score = opportunity_gap × log1p(impressions_90d)`

I used `log1p` (not raw `impressions_90d`) because impressions span a huge range (tens to
hundreds of thousands) while the gap only ranges roughly 0–3 — multiplying directly lets
volume dominate the ranking and drown out the gap itself. The log compresses that wide range
(each ×10 increase in impressions becomes a fixed +constant on the log scale) while still
preserving order. `log1p` specifically (`log(1+x)`) avoids `-inf`/errors for pages with
`impressions_90d = 0`, which matters here since many pages also have `ctr = 0`.

I verified this fix: sorting by `weighted_score` surfaces pages with both a high gap *and*
meaningful traffic — not pages that only look good because of raw volume, and not pages
whose gap is real but statistically unreliable due to tiny sample size.

Because I engineered this myself rather than reading a label FlyRank already computed, I
have to be careful it stays a *proxy* for opportunity, not treated as proven ground truth.


In [5]:
# "no_data" tier pages have avg_position == 0 -- no real ranking signal, exclude them
valid = df[df["position_tier"] != "no_data"].copy()

expected_ctr = valid.groupby("position_tier")["ctr"].transform("mean")
valid["opportunity_gap"] = expected_ctr - valid["ctr"]

valid[["content_id", "position_tier", "ctr", "opportunity_gap"]] \
    .sort_values("opportunity_gap", ascending=False) \
    .head(200)

,content_id,position_tier,ctr,opportunity_gap
6295,content_2d33a679ef65,top_3,0.0,1.483611
6315,content_a005947b4184,top_3,0.0,1.483611
6322,content_71c34ede4c20,top_3,0.0,1.483611
6331,content_a71572e22b2c,top_3,0.0,1.483611
6337,content_7b3578d03752,top_3,0.0,1.483611
...,...,...,...,...
225,content_fca60428d29f,top_3,0.0,1.483611
231,content_1298bc1e12be,top_3,0.0,1.483611
234,content_25a9a6de32d0,top_3,0.0,1.483611
29156,content_7288a4d4c198,top_3,0.0,1.483611


In [6]:
# check granularity of the score: how many distinct opportunity_gap values, and how much repetition?
print("unique opportunity_gap values:", valid["opportunity_gap"].nunique())
print("total rows:", len(valid))

valid["opportunity_gap"].value_counts().head(10)

unique opportunity_gap values: 1052
total rows: 30000


,count
opportunity_gap,
0.652467,4038
0.222484,3425
0.323239,2860
1.483611,1782
0.150212,1107
0.502467,175
0.532467,172
0.492467,170
0.162484,164


In [7]:
# check ctr distribution for the rows sharing the top repeated opportunity_gap value
valid[valid["opportunity_gap"] == 0.652467]["ctr"].describe()


,ctr
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [8]:
# round first, then compare - or use np.isclose
valid["opportunity_gap"].round(6).value_counts().head(10)

,count
opportunity_gap,
0.652467,4038
0.222484,3425
0.323239,2860
1.483611,1782
0.150212,1107
0.502467,175
0.532467,172
0.492467,170
0.162484,164


In [9]:
# now filter using the rounded value
mask = valid["opportunity_gap"].round(6) == 0.652467
valid[mask]["ctr"].describe()

,ctr
count,4038.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


In [10]:
import numpy as np

valid["weighted_score"] = valid["opportunity_gap"] * np.log1p(valid["impressions_90d"])

In [11]:
# sanity check: does the weighting fix the earlier problem?
valid[["content_id", "opportunity_gap", "impressions_90d", "weighted_score"]] \
    .sort_values("weighted_score", ascending=False) \
    .head(10)

,content_id,opportunity_gap,impressions_90d,weighted_score
7678,content_8451fc6f034d,1.453611,272144,18.190613
26844,content_8c19996aa890,1.333611,509252,17.524576
3331,content_4a6607efcb46,1.473611,128068,17.330138
16736,content_e12868d1f396,1.413611,149712,16.845255
9412,content_8053a66bd6ac,1.403611,52687,15.260254
7860,content_d225ec9f3d46,1.433611,26470,14.599610
27107,content_0022a6b4290f,1.413611,29747,14.560919
10893,content_6f81ccd92b64,1.293611,73675,14.498052
20537,content_f4e210ee0c27,1.423611,24784,14.404082
7122,content_7a6df559322d,1.343611,43650,14.355110


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.